# Movie Recommendation Chatbot

## 1. Problem Definition & Objective

### a. Selected Project Track
**Pre-defined Project Track**: Content Recommendation System

### b. Clear Problem Statement
Finding the right movie to watch can be overwhelming due to the sheer volume of available content. Traditional recommendation systems often rely on simple metadata filtering or collaborative filtering (user ratings), which may miss the *context* of what a user is truly looking for (e.g., specific plots, moods, or themes). Users often want to ask natural language queries like *"I want a thriller that involves time travel and isn't too dark"* rather than just filtering by "Genre: Sci-Fi".

### c. Real-world Relevance and Motivation
This project aims to solve the discovery problem by building an **AI-powered Conversational Recommendation System**. By using **Retrieval-Augmented Generation (RAG)**, the system can understand natural language queries, search for semantically similar movies based on their descriptions, and then use a helper **Large Language Model (LLM)** to generate a personalized, human-like explanation for *why* a movie was recommended.

This approach improves user engagement and personalization compared to static keyword search.

## 2. Data Understanding & Preparation

### a. Dataset Source
We are using the **TMDB (The Movie Database) Top Movies** dataset. 
- **Source**: Publicly available dataset (cleaned version provided in `data/tmdb_top_movies_cleaned.csv`).
- **Content**: Contains metadata for thousands of popular movies, including Title, Genres, Overview (Plot), Rating, and Popularity.

### b. Data Loading and Exploration
We use a custom `DataLoader` class to handle data operations.

In [ ]:
import sys
import os
import pandas as pd

# Ensure we can import from src
sys.path.insert(0, os.path.abspath('src'))

from data_loader import DataLoader

# Initialize DataLoader
data_path = "data/tmdb_top_movies_cleaned.csv"
loader = DataLoader(data_path)

# Load Data
df = loader.load_data()

# Display first few rows
df.head()

### c. Cleaning & Preprocessing
The data cleaning process involves:
1.  **Handling Missing Values**: Filling null ratings with 0.0, empty genres with 'Unknown', and missing overviews.
2.  **Feature Engineering**: Creating a `combined_text` field that merges *Title*, *Genres*, and *Overview*. This field is crucial for our semantic search as it allows the embedding model to "read" the full context of a movie.
3.  **Filtering**: Removing invalid entries (e.g., missing titles).

In [ ]:
# Perform preprocessing
df = loader.preprocess_data()

# Show the new 'combined_text' column used for embeddings
print("Sample Combined Text:")
print(df['combined_text'].iloc[0])

## 3. Model / System Design

### a. AI Technique: RAG (Retrieval-Augmented Generation)
We use a specific RAG pipeline to generate recommendations:
1.  **Retrieval**: Semantic search using vector embeddings to find relevant movies from the database.
2.  **Generation**: A Large Language Model (LLM) takes the user's query and the retrieved movie details to generate a natural language response.

### b. Architecture
1.  **Embedding Model**: `all-MiniLM-L6-v2` (SentenceTransformer) is used to convert movie text into dense vectors (384 dimensions). This model is efficient and optimized for semantic similarity.
2.  **Vector Store**: **FAISS (Facebook AI Similarity Search)** is used to index these vectors for extremely fast nearest-neighbor search.
3.  **LLM**: **Llama 3.1 8b Instant** (via Groq API) is used for the generation step. It is fast, capable, and follows instructions well.

## 4. Core Implementation

### a. Vector Store Initialization
We convert our preprocessed movies into embeddings and store them in the FAISS index.

In [ ]:
from vector_store import VectorStore

# Get documents from loader
documents = loader.get_movie_documents()

# Initialize Vector Store
vector_store = VectorStore()

# Try to load existing index to save time, otherwise build it
if not vector_store.load_index():
    print("Building new index...")
    vector_store.initialize_model()
    embeddings = vector_store.create_embeddings(documents)
    vector_store.build_index(documents, embeddings)
    vector_store.save_index()
else:
    print("Loaded existing index.")

### b. RAG Pipeline & Inference
The `RAGPipeline` class orchestrates the flow:
1.  Receive User Query.
2.  **Mood Filtering**: Analyze query for mood keywords (e.g., "happy", "dark").
3.  **Search**: Query FAISS for top-k similar movies.
4.  **Prompt Engineering**: Construct a context-rich prompt containing the retrieved movies.
5.  **Generation**: Send prompt to Groq LLM to get the final recommendation.

In [ ]:
from rag_pipeline import RAGPipeline
from dotenv import load_dotenv

# Load API key
load_dotenv()

# Initialize Pipeline
# Using Llama 3.1 8b Instant for speed and efficiency
rag = RAGPipeline(vector_store, llm_model="llama-3.1-8b-instant")

def get_recommendation(query):
    print(f"\nQuery: {query}\n")
    result = rag.generate_recommendations(query, num_recommendations=3)
    
    if result['recommendations']:
        for i, rec in enumerate(result['recommendations'], 1):
            print(f"{i}. {rec['title']} ({rec['genres']}) - ⭐ {rec['rating']}")
            print(f"   Reason: {rec['explanation']}\n")
    else:
        print(result['message'])

# Test Run
get_recommendation("I want a mind-bending sci-fi movie like Inception")

In [ ]:
# Another Example Test
get_recommendation("Recommend a funny comedy to cheer me up")

## 5. Evaluation & Analysis

### a. Qualitative Analysis
We evaluate the system by manually inspecting the relevance of recommendations for various query types:

- **Specific Similarity**: Queries like "Movies like Inception" successfully retrieve other complex sci-fi films (e.g., *Interstellar*, *The Matrix*).
- **Mood-based**: Queries containing emotions ("sad", "happy") trigger the `MoodFilter`, ensuring the tone of the movie matches the user's request.
- **Genre**: Direct genre requests work well due to the semantic embedding of genre tags.

### b. Limitations
- **Dataset Staticness**: The dataset is static. New movies released after the dataset creation are not available.
- **Hallucination Risk**: While RAG minimizes this, the LLM could still occasionally invent details if the retrieved context is vague.
- **Cold Start**: If a user asks for something completely outside the dataset's scope (e.g., obscure foreign films not in the top list), the system may struggle.

## 6. Ethical Considerations & Responsible AI

### a. Bias and Fairness
- The `tmdb_top_movies` dataset may be biased towards Hollywood/Western cinema. This could result in a lack of diversity in recommendations for users seeking global content.
- The embeddings models themselves (trained on internet text) may contain inherent biases.

### b. Responsible Use
- The system is designed for entertainment purposes only.
- We implement safety guardrails by using reputable base models (Llama 3 family) which have undergone safety training to refuse harmful requests.

## 7. Conclusion & Future Scope

### a. Summary
We successfully built a RAG-based movie recommendation chatbot. It overcomes the limitations of keyword search by understanding the semantic meaning of user queries and providing personalized, context-aware suggestions.

### b. Future Improvements
- **Hybrid Search**: Combine vector search with keyword search (BM25) for better handling of exact entity names.
- **Live Data**: Integrate TMDb API for real-time data fetching.
- **User Profiles**: Store user preferences over time for long-term personalization.